# Lab 9: Streaming Responses — Real-Time Agent UX
---
**SmartClaims AI Agent** | Microsoft Foundry Agent Service (v2.x)

## Objective
Implement token-by-token streaming for better user experience. You'll learn:
- Basic streaming with `stream=True`
- Understanding stream event types and their anatomy
- Streaming with function tool calls (pause → execute → resume)
- Comparing streaming vs blocking performance metrics

## Why Streaming?
Without streaming, users stare at a blank screen until the full response is generated. With streaming:
- **First token appears in ~0.5s** vs. waiting 3-5s for full response
- **Progressive rendering** — users start reading immediately
- **Perceived performance** improves dramatically even if total time is similar

```
Blocking:   [────────────── wait ──────────────] full response
Streaming:  [tok][tok][tok][tok][tok][tok][tok]  progressive
```

## Step 1: Import Dependencies & Create Clients

In [2]:
import sys, os, json, time
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from utils.config import get_clients, MODEL, print_header, print_step
from utils.business_functions import get_claim_status, calculate_fraud_risk
from azure.ai.projects.models import PromptAgentDefinition

project_client, openai_client = get_clients()
print("✅ Clients created")

✅ Clients created


## Step 2: Basic Streaming — Token by Token

The simplest streaming pattern: set `stream=True` and iterate over events. When the event type is `response.output_text.delta`, it contains the next token(s) of the response.

In [4]:
agent = project_client.agents.create_version(
    agent_name="smartclaims-stream",
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are SmartClaims, a helpful insurance assistant. "
            "Keep responses concise (2-3 sentences max)."
        ),
    ),
)
print(f"Agent: {agent.name} v{agent.version}")

Agent: smartclaims-stream v1


In [13]:
query = "What are the top 3 things a customer should do immediately after a car accident? output in 1000 words only"
print(f"👤 User: {query}")
print(f"🤖 Agent: ", end="", flush=True)

stream = openai_client.responses.create(
    extra_body={
        "agent_reference": {
            "name": agent.name,
            "version": agent.version,
            "type": "agent_reference",
        }
    },
    input=query,
    stream=True,
)

for event in stream:
    event_type = event.type if hasattr(event, "type") else type(event).__name__
    if event_type == "response.output_text.delta":
        print(event.delta, end="", flush=True)

print()  # newline after streaming

👤 User: What are the top 3 things a customer should do immediately after a car accident? output in 1000 words only
🤖 Agent: 1. **Ensure Safety**: First, check for injuries and ensure everyone is safe. If possible, move vehicles to a safe location and turn on hazard lights to alert other drivers.

2. **Document the Incident**: Gather information by taking photos of the scene, vehicles, and any visible damages. Exchange contact and insurance details with other drivers involved and collect witness information if available.

3. **Notify Authorities and Insurance**: Call the police to report the accident, especially if there are injuries or significant damage. Notify your insurance company as soon as possible to begin the claims process.


## Step 3: Stream Event Anatomy

Let's analyze all the event types that come through a stream. This helps you understand the streaming protocol and build custom handling.

In [14]:
query2 = "What does comprehensive auto insurance cover?"
print(f"Streaming: \"{query2}\"")
print()

event_counts = {}
first_text_time = None
start_time = time.time()

stream2 = openai_client.responses.create(
    extra_body={
        "agent_reference": {
            "name": agent.name,
            "version": agent.version,
            "type": "agent_reference",
        }
    },
    input=query2,
    stream=True,
)

for event in stream2:
    event_type = event.type if hasattr(event, "type") else type(event).__name__
    event_counts[event_type] = event_counts.get(event_type, 0) + 1
    if event_type == "response.output_text.delta" and first_text_time is None:
        first_text_time = time.time()

end_time = time.time()

print(f"{'Event Type':<50s} | {'Count':>5s}")
print(f"{'-'*50}-+-{'-'*5}")
for etype, count in sorted(event_counts.items()):
    print(f"{etype:<50s} | {count:>5d}")
print()

if first_text_time:
    ttft = first_text_time - start_time
    total = end_time - start_time
    print(f"⏱️  Time to first token: {ttft:.2f}s")
    print(f"⏱️  Total stream time:   {total:.2f}s")

Streaming: "What does comprehensive auto insurance cover?"

Event Type                                         | Count
---------------------------------------------------+------
response.completed                                 |     1
response.content_part.added                        |     1
response.content_part.done                         |     1
response.created                                   |     1
response.in_progress                               |     1
response.output_item.added                         |     1
response.output_item.done                          |     1
response.output_text.delta                         |    43
response.output_text.done                          |     1

⏱️  Time to first token: 1.62s
⏱️  Total stream time:   2.63s


## Step 4: Streaming with Function Calls

When an agent has function tools and decides to call one during streaming, the flow becomes:

```
1. Stream starts → text deltas arrive
2. Agent decides to call a function → function_call event
3. Stream pauses (no more text)
4. Client executes the function locally
5. Client sends result back with previous_response_id
6. Stream resumes → more text deltas with the final answer
```

This is the most complex streaming pattern, but essential for production agents.

In [15]:
# Function tools (reused from Lab 5)
FUNCTION_MAP = {
    "get_claim_status": get_claim_status,
    "calculate_fraud_risk": calculate_fraud_risk,
}

FUNCTION_TOOLS = [
    {
        "type": "function",
        "name": "get_claim_status",
        "description": "Look up the current status of an insurance claim by ID.",
        "parameters": {
            "type": "object",
            "properties": {
                "claim_id": {"type": "string", "description": "Claim ID in CLM-XXXX format"}
            },
            "required": ["claim_id"]
        }
    },
    {
        "type": "function",
        "name": "calculate_fraud_risk",
        "description": "Calculate fraud risk score for a new insurance claim.",
        "parameters": {
            "type": "object",
            "properties": {
                "incident_type": {"type": "string", "enum": ["Auto Collision", "Property Damage", "Medical Claim", "Theft", "Natural Disaster", "Liability", "Fire Damage"]},
                "claim_amount": {"type": "number"},
                "region": {"type": "string", "enum": ["North", "South", "East", "West", "Central"]},
                "days_since_policy_start": {"type": "integer"}
            },
            "required": ["incident_type", "claim_amount", "region", "days_since_policy_start"]
        }
    },
]

agent_fn = project_client.agents.create_version(
    agent_name="smartclaims-stream-fn",
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are SmartClaims Operations Agent. Use your tools "
            "to look up claims and assess fraud risk. Be concise."
        ),
        tools=FUNCTION_TOOLS,
    ),
)
print(f"Agent: {agent_fn.name} v{agent_fn.version}")

Agent: smartclaims-stream-fn v1


In [17]:
query3 = "Look up claim CLM-0001 and tell me its status."
print(f"👤 User: {query3}")
print(f"🤖 Agent: ", end="", flush=True)

conv = openai_client.conversations.create()
stream3 = openai_client.responses.create(
    conversation=conv.id,
    extra_body={
        "agent_reference": {
            "name": agent_fn.name,
            "version": agent_fn.version,
            "type": "agent_reference",
        }
    },
    input=query3,
    stream=True,
)

# Collect events — extract function calls from the completed response
function_calls = []
response_id = None

for event in stream3:
    event_type = event.type if hasattr(event, "type") else type(event).__name__

    if event_type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event_type == "response.function_call_arguments.done":
        print(f"\n   🔧 [TOOL CALL] {event.name}({event.arguments})")
    elif event_type == "response.completed":
        response_id = event.response.id if hasattr(event, "response") else None
        # Extract function calls with call_id from the completed response output
        if hasattr(event, "response") and event.response.output:
            for item in event.response.output:
                if item.type == "function_call":
                    function_calls.append({
                        "name": item.name,
                        "arguments": item.arguments,
                        "call_id": item.call_id,
                    })

# Execute functions and continue streaming
if function_calls and response_id:
    tool_outputs = []
    for fc in function_calls:
        func = FUNCTION_MAP.get(fc["name"])
        if func:
            args = json.loads(fc["arguments"])
            result = func(**args)
        else:
            result = json.dumps({"error": f"Unknown: {fc['name']}"})
        tool_outputs.append({
            "type": "function_call_output",
            "call_id": fc["call_id"],
            "output": result,
        })

    print(f"   📤 [TOOL RESULT] Sending results back...")
    print(f"🤖 Agent: ", end="", flush=True)

    stream4 = openai_client.responses.create(
        extra_body={
            "agent_reference": {
                "name": agent_fn.name,
                "version": agent_fn.version,
                "type": "agent_reference",
            }
        },
        input=tool_outputs,
        previous_response_id=response_id,
        stream=True,
    )

    for event in stream4:
        event_type = event.type if hasattr(event, "type") else type(event).__name__
        if event_type == "response.output_text.delta":
            print(event.delta, end="", flush=True)

    print()

👤 User: Look up claim CLM-0001 and tell me its status.
🤖 Agent: 
   🔧 [TOOL CALL] None({"claim_id":"CLM-0001"})
   📤 [TOOL RESULT] Sending results back...
🤖 Agent: The status of claim **CLM-0001** is **Denied**. Here are the details:

- **Policy Number:** POL-719176
- **Policyholder:** Customer_0001
- **Incident Type:** Liability
- **Incident Date:** 2024-10-23
- **Claim Amount:** $32,209.90
- **Approved Amount:** $0.00
- **Adjuster:** Lisa Anderson
- **Fraud Score:** 0.36
- **Region:** East
- **Processing Days:** 14


## Step 5: Streaming vs Blocking — Performance Comparison

Let's measure the actual difference in time-to-first-token between streaming and blocking modes.

In [ ]:
comparison_query = (
    "Explain the difference between comprehensive and "
    "collision auto insurance coverage in 2 sentences."
)
print(f"Query: \"{comparison_query[:60]}...\"")
print()

# Blocking (non-streaming)
t0 = time.time()
blocking_resp = openai_client.responses.create(
    extra_body={
        "agent_reference": {
            "name": agent.name, "version": agent.version,
            "type": "agent_reference",
        }
    },
    input=comparison_query,
)
blocking_time = time.time() - t0
blocking_text = blocking_resp.output_text

# Streaming
t0 = time.time()
streaming_first_token = None
streaming_text = ""

stream5 = openai_client.responses.create(
    extra_body={
        "agent_reference": {
            "name": agent.name, "version": agent.version,
            "type": "agent_reference",
        }
    },
    input=comparison_query,
    stream=True,
)

for event in stream5:
    event_type = event.type if hasattr(event, "type") else type(event).__name__
    if event_type == "response.output_text.delta":
        if streaming_first_token is None:
            streaming_first_token = time.time() - t0
        streaming_text += event.delta

streaming_total = time.time() - t0

print(f"| Metric                 | Blocking     | Streaming    |")
print(f"|------------------------|--------------|--------------|")
print(f"| Time to first token    | {blocking_time:.2f}s (full) | {streaming_first_token:.2f}s        |")
print(f"| Total response time    | {blocking_time:.2f}s        | {streaming_total:.2f}s        |")
print(f"| Response length        | {len(blocking_text):>5d} chars | {len(streaming_text):>5d} chars |")
print()
print("💡 Streaming delivers first token faster → better perceived UX")

## Step 6: Clean Up

In [ ]:
for name in ["smartclaims-stream", "smartclaims-stream-fn"]:
    try:
        project_client.agents.delete(name)
        print(f"✅ Deleted: {name}")
    except Exception:
        print(f"⏭️  Skip: {name}")

## ✅ Lab 9 Complete!

### Key Takeaways
| Concept | What You Learned |
|---------|------------------|
| `stream=True` | Enable token-by-token streaming |
| Event types | `response.output_text.delta`, `response.completed`, etc. |
| Function streaming | Pause stream → execute function → resume stream |
| Time-to-first-token | Key UX metric — streaming wins significantly |
| `previous_response_id` | Continue streaming after function call results |

---
**Next →** [Lab 10: Azure AI Search](lab10_azure_ai_search.ipynb) — Enterprise-grade RAG with Azure AI Search